# 模块 6 节点壳：视觉 PID ROS 适配示例

对应 `pid_controller_node.py`，仅演示 `VisualTrackArray -> VisualTrack`、纯 `PositionCommand -> PositionTarget`，以及话题、frame、mask 和 50 Hz 接口。

> **运行前提**：使用 **ROS Noetic (Py3.8)** 内核，并已 `source /opt/ros/noetic/setup.bash`、`source catkin_ws/devel/setup.bash`及 `catkin_make`。本 notebook 不初始化或运行节点，不发布任何话题，不调用 `spin()`。

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while root != root.parent and not (root / 'catkin_ws').is_dir():
    root = root.parent
scripts = root / 'catkin_ws' / 'src' / 'uav_tracking' / 'scripts'
assert scripts.is_dir(), '未找到 ROS 工作区'
sys.path.insert(0, str(scripts))

from mavros_msgs.msg import PositionTarget
from uav_tracking.msg import VisualTrack as VisualTrackMsg, VisualTrackArray
import pid_controller_node as node
from pid_controller import on_timer as pure_on_timer

print('input :', node.TOPIC_TRACKS, 'uav_tracking/VisualTrackArray')
print('output:', node.TOPIC_OUTPUT, 'mavros_msgs/PositionTarget')
print('rate  : 50 Hz')
assert node.TOPIC_OUTPUT == '/ctrl/visual'

## 1. ROS 视觉航迹拆包

节点把 `VisualTrack.msg` 转为公共纯类型 `VisualTrack`，不在适配层重复 PID 算法。

In [ ]:
array = VisualTrackArray()
for track_id, bbox, area in (
    (1, [270.0, 190.0, 100.0, 100.0], 0.10),
    (2, [430.0, 310.0, 100.0, 100.0], 0.03),
):
    item = VisualTrackMsg()
    item.id, item.bbox, item.area_ratio = track_id, bbox, area
    array.tracks.append(item)
tracks = node.tracks_from_msg(array)
print('pure tracks:', tracks)
assert node.tracks_from_msg(VisualTrackArray()) == []

## 2. `PositionCommand -> PositionTarget`

纯算法选择面积最大的目标并生成指令；节点只复制速度和偏航速率字段。

In [ ]:
command = pure_on_timer(tracks, valid_age=0.0, last_cmd=None, dt=0.02)
target = node.command_to_position_target(command)
print('pure command  :', command)
print('PositionTarget: frame=%d mask=%d velocity=(%+.3f, %+.3f, %+.3f) yaw_rate=%+.3f' %
      (target.coordinate_frame, target.type_mask, target.velocity.x,
       target.velocity.y, target.velocity.z, target.yaw_rate))
assert target.coordinate_frame == PositionTarget.FRAME_BODY_OFFSET_NED == 9
assert target.type_mask == node.VELOCITY_TYPE_MASK == 1479
assert target.velocity.y == 0.0
assert node.TOPIC_OUTPUT != '/mavros/setpoint_raw/local'

## 3. 超时零输出的消息形式

距最后有效输入超过 0.2 s 后，纯算法返回零速度指令，节点仍以同一 frame/mask 装包。

In [ ]:
stopped_command = pure_on_timer([], valid_age=0.21, last_cmd=command, dt=0.02)
stopped = node.command_to_position_target(stopped_command)
print('stopped:', (stopped.velocity.x, stopped.velocity.y, stopped.velocity.z, stopped.yaw_rate))
assert (stopped.velocity.x, stopped.velocity.y, stopped.velocity.z, stopped.yaw_rate) == (0.0, 0.0, 0.0, 0.0)
assert stopped.coordinate_frame == 9 and stopped.type_mask == 1479